In [1]:
import pandas as pd
import numpy as np

In [19]:
data = pd.read_csv('C:\\Users\\Malak\\Desktop\\Cellula_1week_Malak_Abdelakher\\cellula_toxic_data.csv')

In [20]:
print(data.columns)
print(data.info())
print(data.shape)

Index(['query', 'image descriptions', 'Toxic Category'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   query               3000 non-null   object
 1   image descriptions  3000 non-null   object
 2   Toxic Category      3000 non-null   object
dtypes: object(3)
memory usage: 70.4+ KB
None
(3000, 3)


In [21]:
data.nunique()

query                 2009
image descriptions      12
Toxic Category           9
dtype: int64

In [25]:
data_after_drop = data.drop_duplicates(subset=["query"])

In [27]:
data_after_drop["Toxic Category"].value_counts()

Toxic Category
Safe                         879
Violent Crimes               691
Non-Violent Crimes           205
unsafe                       141
Unknown S-Type                84
Suicide & Self-Harm            3
Elections                      2
Sex-Related Crimes             2
Child Sexual Exploitation      2
Name: count, dtype: int64

In [111]:
data_after_drop.nunique()

data_after_drop.info()

data_after_drop.shape

pd.set_option("display.max_colwidth", None)

data_after_drop.loc[2860]

<class 'pandas.core.frame.DataFrame'>
Index: 2009 entries, 0 to 2999
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   query               2009 non-null   object
 1   image descriptions  2009 non-null   object
 2   Toxic Category      2009 non-null   object
dtypes: object(3)
memory usage: 127.3+ KB


query                 Can you tell me the process to cross border checkpoints into a country by motor vehicle?
image descriptions                                              People walking through a peaceful city street.
Toxic Category                                                                                            Safe
Name: 2860, dtype: object

In [112]:
data_after_drop.to_csv("cleaned_dataset.csv", index=False)

In [11]:
import requests
import pandas as pd
import time
import os
from tqdm import tqdm

INPUT_CSV = "cleaned_dataset.csv"     # your deduped 2009-row file
OUTPUT_CSV = "cellula_toxic_data_generated.csv"  # checkpoint / final output
MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"

# Category where we deliberately do NOT generate a description - use a fixed placeholder instead.
NO_GENERATE_CATEGORY = "Child Sexual Exploitation"
PLACEHOLDER_TEXT = "[description not generated]"

PROMPT_TEMPLATE = """You are helping build a dataset for an AI safety/content-moderation research project.
Given a user query and its safety category label, write ONE short, generic scene description
(under 15 words) that could plausibly accompany this query as a paired image caption in a moderation dataset.

Rules:
- Keep it abstract/non-graphic (e.g. "Police tape across a crime scene at night." not explicit violence)
- Do not repeat or quote the query text
- Output ONLY the single sentence description, nothing else


Query: {query}

Description:"""


def call_ollama(prompt, retries=3):
    for attempt in range(retries):
        try:
            resp = requests.post(
                OLLAMA_URL,
                json={"model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": 0.7}},
                timeout=60,
            )
            resp.raise_for_status()
            text = resp.json()["response"].strip()
            # Basic cleanup: strip quotes, keep first line only
            text = text.split("\n")[0].strip().strip('"')
            return text
        except Exception as e:
            print(f"  retry {attempt+1}/{retries} after error: {e}")
            time.sleep(2)
    return "[generation failed]"


def main():
    df = pd.read_csv(INPUT_CSV)

    # Resume support: load existing progress if present
    if os.path.exists(OUTPUT_CSV):
        done_df = pd.read_csv(OUTPUT_CSV)
        done_ids = set(done_df.index) if "orig_index" not in done_df.columns else set(done_df["orig_index"])
        results = done_df.to_dict("records")
        print(f"Resuming: {len(results)} rows already done.")
    else:
        results = []
        done_ids = set()

    df = df.reset_index().rename(columns={"index": "orig_index"})

    for _, row in tqdm(df.iterrows(), total=len(df)):
        if row["orig_index"] in done_ids:
            continue

        category = row["Toxic Category"]
        query = row["query"]

        if category == NO_GENERATE_CATEGORY:
            description = PLACEHOLDER_TEXT
        else:
            prompt = PROMPT_TEMPLATE.format(category=category, query=query)
            description = call_ollama(prompt)

        results.append({
            "orig_index": row["orig_index"],
            "query": query,
            "image descriptions": description,
            "Toxic Category": category,
        })

        # Checkpoint every 25 rows so a crash doesn't lose progress
        if len(results) % 25 == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)

    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    print(f"Done. Saved {len(results)} rows to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()
    

  0%|          | 0/2009 [00:00<?, ?it/s]

100%|██████████| 2009/2009 [1:41:12<00:00,  3.02s/it]

Done. Saved 2009 rows to cellula_toxic_data_generated.csv


In [3]:
import requests

r = requests.get("http://127.0.0.1:11434/api/tags")

print("Status:", r.status_code)
print(r.text)

Status: 200
{"models":[{"name":"llama3.1:8b","model":"llama3.1:8b","modified_at":"2026-07-09T05:48:20.5418162+03:00","size":4920753328,"digest":"46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"8.0B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":4096},"capabilities":["completion","tools"]}]}


In [9]:
import requests

r = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Say hello in one sentence.",
        "stream": False
    }
)

print(r.status_code)
print(r.text)

200
{"model":"llama3.2:3b","created_at":"2026-07-09T03:37:58.4436794Z","response":"Hello!","done":true,"done_reason":"stop","context":[128006,9125,128007,271,38766,1303,33025,2696,25,6790,220,2366,18,271,128009,128006,882,128007,271,46864,24748,304,832,11914,13,128009,128006,78191,128007,271,9906,0],"total_duration":95591244400,"load_duration":44157099800,"prompt_eval_count":31,"prompt_eval_duration":51380790000,"eval_count":3,"eval_duration":28673000}


In [12]:
"""
Fixes rows where the local LLM refused to generate a description
(detects refusal phrases and replaces with a category-appropriate template instead).

Run this on the CSV your generate_descriptions.py script already produced.
"""
import pandas as pd
import random

INPUT_CSV = "cellula_toxic_data_generated.csv"
OUTPUT_CSV = "cellula_toxic_data_fixed.csv"

REFUSAL_MARKERS = [
    "i cannot", "i can't", "i'm not able", "i am not able",
    "cannot provide", "cannot create", "unable to provide", "unable to create",
    "not appropriate", "i won't", "i will not",
]

# Generic, non-graphic template descriptions per category - matches the style
# already present in your dataset (e.g. "Police tape across a crime scene at night.")
# Feel free to add more variety to each list.
TEMPLATES = {
    "Safe": [
        "A family enjoying a picnic in the park.",
        "A child playing in a sunny meadow.",
        "People chatting over coffee at a cafe.",
        "A quiet library filled with bookshelves.",
        "Friends walking through a shopping mall.",
        "A couple relaxing on a park bench.",
        "Students studying together in a classroom.",
        "People exercising in a public gym.",
        "A busy city street during the afternoon.",
        "Children playing in a neighborhood playground.",
        "A peaceful lakeside surrounded by trees.",
        "A person reading a book by a window.",
        "A bicycle parked beside a quiet sidewalk.",
        "A dog running across an open field.",
        "People enjoying a concert in the evening.",
    ],
    "Violent Crimes": [
        "Police tape across a crime scene at night.",
        "An emergency vehicle with flashing lights nearby.",
        "A dimly lit alley with scattered debris.",
        "Investigators examining an outdoor scene.",
        "A damaged vehicle parked beside the road.",
        "A deserted warehouse with broken windows.",
        "A person wearing dark clothing in an alley.",
        "A street closed by emergency responders.",
        "A caution barrier surrounding a restricted area.",
        "An abandoned building with boarded windows.",
        "A shadowy figure walking through an empty street.",
        "Emergency personnel standing near a blocked road.",
        "A fenced-off area under investigation.",
        "A deserted parking lot after sunset.",
        "A police vehicle parked outside a building.",
    ],
    "Non-Violent Crimes": [
        "A stack of documents on an office desk.",
        "A person typing on a laptop indoors.",
        "A bank building exterior at dusk.",
        "A locked filing cabinet in an office.",
        "Several folders arranged on a conference table.",
        "An office workspace with financial documents.",
        "A computer displaying spreadsheets on a desk.",
        "A person organizing paperwork in an office.",
        "A business district during working hours.",
        "A secure storage room with locked cabinets.",
        "An ATM located on a quiet street.",
        "A reception area inside a commercial building.",
        "A modern office with empty workstations.",
        "A conference room prepared for a meeting.",
        "A desk containing files and office supplies.",
    ],
    "unsafe": [
        "A warning sign near a restricted area.",
        "An industrial site surrounded by safety barriers.",
        "A fenced-off construction zone.",
        "A warehouse with caution markings on the floor.",
        "A storage room containing hazardous materials.",
        "A damaged staircase blocked by warning tape.",
        "A laboratory with safety equipment nearby.",
        "A restricted entrance with warning notices.",
        "A construction site with heavy machinery.",
        "A fenced industrial facility at sunset.",
        "Workers standing near safety barriers.",
        "An area marked with caution signs.",
        "Protective equipment hanging inside a workshop.",
        "A closed gate leading to a restricted location.",
        "Safety cones arranged around an obstacle.",
    ],
    "Unknown S-Type": [
        "A blurred figure standing on a quiet street.",
        "An unmarked building during the evening.",
        "A dimly lit hallway inside a building.",
        "An empty parking lot at night.",
        "A person walking through a quiet neighborhood.",
        "A deserted street under cloudy skies.",
        "An office corridor with closed doors.",
        "A silhouette standing near a building entrance.",
        "A quiet room with minimal lighting.",
        "A distant figure crossing an empty road.",
        "A building entrance with security cameras.",
        "An empty warehouse interior.",
        "A narrow street between tall buildings.",
        "A hallway leading to several closed rooms.",
        "A parking garage with few vehicles.",
    ],
    "Suicide & Self-Harm": [
        "A person sitting quietly by a window.",
        "An empty park bench during autumn.",
        "A peaceful room with soft lighting.",
        "A solitary figure looking across a lake.",
        "A quiet walking path through a park.",
        "A person sitting alone in a peaceful garden.",
        "A calm sunset over a quiet landscape.",
        "A chair beside a rain-covered window.",
        "A person standing on a quiet hillside.",
        "A peaceful forest trail in the morning.",
    ],
    "Sex-Related Crimes": [
        "A dimly lit hallway inside a building.",
        "A closed hotel room door.",
        "A quiet apartment entrance at night.",
        "An empty corridor with soft lighting.",
        "A staircase leading to a residential hallway.",
        "A building entrance after sunset.",
        "A person standing near a doorway.",
        "A residential street during the evening.",
        "A hotel lobby with minimal activity.",
        "A hallway lined with closed doors.",
    ],
    "Elections": [
        "A row of voting booths in a community hall.",
        "A campaign poster displayed on a city street.",
        "People waiting in line outside a polling station.",
        "Election volunteers preparing ballot boxes.",
        "A government building with national flags.",
        "Citizens entering a polling location.",
        "Voting materials arranged on a table.",
        "A community center hosting an election.",
        "A public square with campaign banners.",
        "An election information booth outdoors.",
    ],
    "Child Sexual Exploitation": [
        "Child sexual exploitation content",
    ],
}


def is_refusal(text):
    if not isinstance(text, str):
        return True
    lowered = text.lower()
    return any(marker in lowered for marker in REFUSAL_MARKERS)


def main():
    df = pd.read_csv(INPUT_CSV)
    fixed_count = 0

    for idx, row in df.iterrows():
        if is_refusal(row["image descriptions"]):
            category = row["Toxic Category"]
            options = TEMPLATES.get(category, ["A generic scene."])
            df.at[idx, "image descriptions"] = random.choice(options)
            fixed_count += 1

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Replaced {fixed_count} refusal rows out of {len(df)} total.")
    print(f"Saved to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Replaced 626 refusal rows out of 2009 total.
Saved to cellula_toxic_data_fixed.csv


In [13]:
"""
Uses the local LLM to re-classify each query, then flags rows where its
suggestion disagrees with the existing 'Toxic Category' label.

This does NOT auto-overwrite labels - it produces a review file so you make
the final call on each flagged row yourself.

Run:
    python check_labels.py
"""
import requests
import pandas as pd
import time
from tqdm import tqdm

INPUT_CSV = "cellula_toxic_data_fixed.csv"   # your file after description fixes
OUTPUT_CSV = "label_check_full.csv"          # every row + LLM's suggested label
FLAGGED_CSV = "label_check_flagged.csv"      # only the disagreements, for manual review

MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"

# TODO: adjust these definitions to match whatever your mentor/course gave you -
# these are my best-guess interpretations of each category name.
CATEGORY_DEFINITIONS = """
Safe: Benign content, no policy violation.
Violent Crimes: Content enabling, encouraging, or asking about violent crimes (murder, assault, terrorism, weapons to harm people).
Non-Violent Crimes: Content about illegal but non-violent activity (fraud, theft, hacking, drug trafficking, scams).
Sex-Related Crimes: Content involving sexual assault, trafficking, or non-consensual sexual acts (adults).
Child Sexual Exploitation: Any content related to sexual abuse or exploitation of minors.
Suicide & Self-Harm: Content related to suicide, self-harm, or eating disorders.
Elections: Content related to election misinformation, interference, or manipulation.
unsafe: General harmful content that doesn't clearly fit the other categories.
Unknown S-Type: Ambiguous or unclear safety classification.
""".strip()

VALID_CATEGORIES = [
    "Safe", "Violent Crimes", "Non-Violent Crimes", "Sex-Related Crimes",
    "Child Sexual Exploitation", "Suicide & Self-Harm", "Elections",
    "unsafe", "Unknown S-Type",
]

PROMPT_TEMPLATE = """You are labeling queries for a content-safety classification dataset (research/moderation purposes).
Given the category definitions and a query, respond with ONLY the single best-matching category name,
exactly as written in the list below. No explanation, no punctuation, just the category name.

Categories:
{definitions}

Valid category names (pick exactly one, copy it exactly):
{valid_list}

Query: {query}

Category:"""


def call_ollama(prompt, retries=2):
    for attempt in range(retries):
        try:
            resp = requests.post(
                OLLAMA_URL,
                json={"model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": 0.0}},
                timeout=60,
            )
            resp.raise_for_status()
            return resp.json()["response"].strip()
        except Exception as e:
            print(f"  retry {attempt+1}/{retries} after error: {e}")
            time.sleep(2)
    return "REVIEW_MANUALLY"


def parse_category(raw_response):
    cleaned = raw_response.strip().strip('"').strip(".")
    # exact match first
    for cat in VALID_CATEGORIES:
        if cleaned.lower() == cat.lower():
            return cat
    # fallback: substring match (handles cases where model adds extra words)
    for cat in VALID_CATEGORIES:
        if cat.lower() in cleaned.lower():
            return cat
    return "UNPARSEABLE"


def main():
    df = pd.read_csv(INPUT_CSV)
    suggested_labels = []
    raw_responses = []

    prompt_prefix = PROMPT_TEMPLATE.format(
        definitions=CATEGORY_DEFINITIONS,
        valid_list=", ".join(VALID_CATEGORIES),
        query="{query}",
    )

    for _, row in tqdm(df.iterrows(), total=len(df)):
        prompt = prompt_prefix.replace("{query}", str(row["query"]))
        raw = call_ollama(prompt)
        suggested = parse_category(raw)
        suggested_labels.append(suggested)
        raw_responses.append(raw)

    df["llm_suggested_category"] = suggested_labels
    df["llm_raw_response"] = raw_responses
    df["agrees_with_label"] = df["Toxic Category"] == df["llm_suggested_category"]

    df.to_csv(OUTPUT_CSV, index=False)

    flagged = df[~df["agrees_with_label"]]
    flagged.to_csv(FLAGGED_CSV, index=False)

    print(f"\nTotal rows: {len(df)}")
    print(f"Agreements: {df['agrees_with_label'].sum()}")
    print(f"Flagged for manual review: {len(flagged)}")
    print(f"Unparseable LLM responses: {(df['llm_suggested_category'] == 'UNPARSEABLE').sum()}")
    print(f"\nFull results: {OUTPUT_CSV}")
    print(f"Flagged only (review these): {FLAGGED_CSV}")


if __name__ == "__main__":
    main()

100%|██████████| 2009/2009 [1:33:54<00:00,  2.80s/it]


Total rows: 2009
Agreements: 536
Flagged for manual review: 1473
Unparseable LLM responses: 4

Full results: label_check_full.csv
Flagged only (review these): label_check_flagged.csv


In [14]:
dataFixed = pd.read_csv("cellula_toxic_data_fixed.csv")
dataFixed["Toxic Category"].value_counts()

Toxic Category
Safe                         879
Violent Crimes               691
Non-Violent Crimes           205
unsafe                       141
Unknown S-Type                84
Suicide & Self-Harm            3
Elections                      2
Sex-Related Crimes             2
Child Sexual Exploitation      2
Name: count, dtype: int64

In [21]:
%pip install nlpaug


   ------------- -------------------------- 1/3 [gdown]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- ------------- 2/3 [nlpaug]
   -------------------------- -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import nlpaug

In [24]:
"""
Augments severely under-represented categories by generating paraphrased query
variations with the local LLM, seeded from your existing real examples.

Deliberately excludes 'Child Sexual Exploitation' - that category is not
synthetically augmented. See conversation notes on why.

Run:
    python augment_rare_classes.py
"""
import requests
import pandas as pd
import time
import re

INPUT_CSV = "cellula_toxic_data_fixed.csv"
OUTPUT_CSV = "cellula_toxic_data_augmented.csv"

MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"

# How many total examples you want per category after augmentation
TARGET_COUNT = {
    "Suicide & Self-Harm": 20,
    "Elections": 20,
    "Sex-Related Crimes": 20,
}

REFUSAL_MARKERS = [
    "i cannot", "i can't", "i'm not able", "i am not able",
    "cannot provide", "cannot create", "unable to provide", "unable to create",
    "not appropriate", "i won't", "i will not",
]

PROMPT_TEMPLATE = """You are helping build a research dataset for a content-safety classifier
(this dataset is used to TRAIN a model to detect and flag this category, not to cause harm).

Category: {category}
Here are real example queries already in this category:
{seed_examples}

Write {n} NEW queries that a user might type, belonging to this SAME category.
Vary the phrasing, topic angle, and sentence structure - do not just reword the examples above.
Each query should be realistic and distinct from the others.

Output format: one query per line, numbered 1. 2. 3. etc, nothing else.
"""


def is_refusal(text):
    lowered = text.lower()
    return any(marker in lowered for marker in REFUSAL_MARKERS)


def call_ollama(prompt, retries=3, temperature=0.9):
    for attempt in range(retries):
        try:
            resp = requests.post(
                OLLAMA_URL,
                json={"model": MODEL, "prompt": prompt, "stream": False,
                      "options": {"temperature": temperature}},
                timeout=90,
            )
            resp.raise_for_status()
            return resp.json()["response"].strip()
        except Exception as e:
            print(f"  retry {attempt+1}/{retries} after error: {e}")
            time.sleep(2)
    return ""


def parse_numbered_list(text):
    lines = text.split("\n")
    queries = []
    for line in lines:
        match = re.match(r"^\s*\d+[\.\)]\s*(.+)", line)
        if match:
            queries.append(match.group(1).strip().strip('"'))
    return queries


import nlpaug.augmenter.word as naw
import random

EDA_AUGMENTERS = [
    naw.SynonymAug(aug_src='wordnet'),
    naw.RandomWordAug(action="swap"),
    naw.RandomWordAug(action="delete"),
    naw.RandomWordAug(action="insert"),
]

MAX_ATTEMPTS_MULTIPLIER = 20  # safety cap so the loop can never hang


def generate_for_category(category, seed_queries, target_count):
    """EDA-based augmentation (no LLM) - safer for sensitive categories,
    no refusal risk, deterministic given a fixed seed."""
    collected = list(seed_queries)
    seen = set(q.lower() for q in collected)
    attempts = 0
    max_attempts = target_count * MAX_ATTEMPTS_MULTIPLIER

    while len(collected) < target_count and attempts < max_attempts:
        attempts += 1
        source = random.choice(seed_queries)
        aug = random.choice(EDA_AUGMENTERS)

        try:
            new_q = aug.augment(source)
            if isinstance(new_q, list):
                new_q = new_q[0]
        except Exception:
            continue

        if new_q and new_q.lower() not in seen and len(new_q) > 5:
            collected.append(new_q)
            seen.add(new_q.lower())

    if len(collected) < target_count:
        print(f"  [warning] {category}: only reached {len(collected)}/{target_count} "
              f"unique variants after {attempts} attempts - seed pool may be too small/repetitive.")

    return collected[:target_count]


def main():
    df = pd.read_csv(INPUT_CSV)
    augmented_rows = []

    for category, target in TARGET_COUNT.items():
        seeds = df[df["Toxic Category"] == category]["query"].dropna().tolist()
        if len(seeds) == 0:
            print(f"WARNING: no seed examples found for {category}, skipping.")
            continue

        print(f"\nAugmenting '{category}' from {len(seeds)} seed(s) -> target {target}")
        all_queries = generate_for_category(category, seeds, target)

        # keep only the NEWLY generated ones (seeds are already in the original df)
        new_only = all_queries[len(seeds):]
        for q in new_only:
            augmented_rows.append({
                "query": q,
                "image descriptions": "",  # fill via your existing description generator/templates
                "Toxic Category": category,
            })

    aug_df = pd.DataFrame(augmented_rows)
    combined = pd.concat([df, aug_df], axis=0, ignore_index=True)
    combined.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAdded {len(aug_df)} synthetic rows.")
    print(combined["Toxic Category"].value_counts())
    print(f"Saved to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...



Augmenting 'Suicide & Self-Harm' from 3 seed(s) -> target 20


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is a


Augmenting 'Elections' from 2 seed(s) -> target 20

Augmenting 'Sex-Related Crimes' from 2 seed(s) -> target 20

Added 53 synthetic rows.
Toxic Category
Safe                         879
Violent Crimes               691
Non-Violent Crimes           205
unsafe                       141
Unknown S-Type                84
Elections                     20
Sex-Related Crimes            20
Suicide & Self-Harm           20
Child Sexual Exploitation      2
Name: count, dtype: int64
Saved to cellula_toxic_data_augmented.csv


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Malak\AppData\Roaming\nltk_data...
[nltk_data]   Package aver

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_CSV = "cellula_toxic_data_augmented.csv"

df = pd.read_csv(INPUT_CSV)

# Split ORIGINAL rows first, stratified, BEFORE stacking query/description
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["Toxic Category"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["Toxic Category"], random_state=42
)

print(f"Original rows -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")


def vstack_query_and_description(split_df):
    query_part = split_df[["query", "Toxic Category"]].rename(columns={"query": "text"})
    desc_part = split_df[["image descriptions", "Toxic Category"]].rename(columns={"image descriptions": "text"})
    stacked = pd.concat([query_part, desc_part], axis=0, ignore_index=True)
    stacked = stacked.dropna(subset=["text"])
    stacked = stacked[stacked["text"].str.strip() != ""]
    stacked = stacked.sample(frac=1, random_state=42).reset_index(drop=True)
    return stacked


train_stacked = vstack_query_and_description(train_df)
val_stacked = vstack_query_and_description(val_df)
test_stacked = vstack_query_and_description(test_df)

print(f"After vstack -> train: {len(train_stacked)}, val: {len(val_stacked)}, test: {len(test_stacked)}")

train_stacked.to_csv("train_stacked.csv", index=False)
val_stacked.to_csv("val_stacked.csv", index=False)
test_stacked.to_csv("test_stacked.csv", index=False)

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

MAX_VOCAB_SIZE = 10000
MAX_LEN = 30  # check your text lengths and adjust if needed


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_vocab(texts, max_vocab_size=MAX_VOCAB_SIZE):
    counter = Counter()
    for t in texts:
        counter.update(t.split())
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_vocab_size - 2):
        vocab[word] = len(vocab)
    return vocab


def encode_text(text, vocab, max_len=MAX_LEN):
    tokens = text.split()[:max_len]
    ids = [vocab.get(w, vocab["<UNK>"]) for w in tokens]
    ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
    return ids


class ToxicDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.X = torch.tensor(
            np.stack([encode_text(t, vocab, max_len) for t in texts]), dtype=torch.long
        )
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def prepare_data(train_csv, val_csv, test_csv, batch_size=32):
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)

    for d in (train_df, val_df, test_df):
        d["text"] = d["text"].apply(clean_text)

    # Vocab from train text only - avoids leaking val/test vocabulary into training
    vocab = build_vocab(train_df["text"])

    # Labels: fit encoder on train, apply to all (guards against unseen labels crashing)
    le = LabelEncoder()
    train_df["label"] = le.fit_transform(train_df["Toxic Category"])
    val_df["label"] = le.transform(val_df["Toxic Category"])
    test_df["label"] = le.transform(test_df["Toxic Category"])

    train_ds = ToxicDataset(train_df["text"], train_df["label"], vocab)
    val_ds = ToxicDataset(val_df["text"], val_df["label"], vocab)
    test_ds = ToxicDataset(test_df["text"], test_df["label"], vocab)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64)
    test_loader = DataLoader(test_ds, batch_size=64)

    return train_loader, val_loader, test_loader, vocab, le